In [8]:
import json
import pandas as pd

def jprint(obj):
    print(json.dumps(obj, indent=2))


In [5]:
with open('../outputs/predictions-ragent-Qwen2.5-1.5B-Instruct-musique-grpo.jsonl') as f:
    records = [json.loads(line) for line in f]
print(len(records))

200


In [17]:
df = pd.DataFrame(records)
df.head()

,answer,n_hops,prompt,docs,answers,supporting_titles,trajectory
0,Casa Loma,3,[{'content': 'Answer the question based on the...,"[{'id': 4, 'is_supporting': False, 'text': '# ...",[Casa Loma],"[Sekou Lumumba, Casa Loma, Fall (Serena Ryder ...",[{'content': 'Answer the question based on the...
1,Yuan Shikai,3,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Yuan Shikai],[Sino-Tibetan relations during the Ming dynast...,[{'content': 'Answer the question based on the...
2,Tualatin Mountains,3,[{'content': 'Answer the question based on the...,"[{'id': 5, 'is_supporting': True, 'text': '# P...",[Tualatin Mountains],"[Portland, Oregon, Coles Creek (Pennsylvania),...",[{'content': 'Answer the question based on the...
3,Casa Loma,3,[{'content': 'Answer the question based on the...,"[{'id': 1, 'is_supporting': False, 'text': '# ...",[Casa Loma],"[Arna Selznick, Casa Loma, Natalie Turner]",[{'content': 'Answer the question based on the...
4,1941,3,[{'content': 'Answer the question based on the...,"[{'id': 2, 'is_supporting': False, 'text': '# ...",[1941],"[Pacific War, Bang Bon District, Ercole Manfredi]",[{'content': 'Answer the question based on the...


In [18]:
from verifiers.rubrics.utils import get_last_answer

df['predicted_answer'] = df['trajectory'].apply(get_last_answer)

In [20]:
from verifiers.metrics.musique import compute_metrics, exact_match, f1

df['exact_match'] = df.apply(lambda row: exact_match(row['predicted_answer'], row['answers']), axis=1)
df['f1'] = df.apply(lambda row: f1(row['predicted_answer'], row['answers']), axis=1)

df.head()

,answer,n_hops,prompt,docs,answers,supporting_titles,trajectory,predicted_answer,exact_match,f1
0,Casa Loma,3,[{'content': 'Answer the question based on the...,"[{'id': 4, 'is_supporting': False, 'text': '# ...",[Casa Loma],"[Sekou Lumumba, Casa Loma, Fall (Serena Ryder ...",[{'content': 'Answer the question based on the...,Toronto,0,0.0
1,Yuan Shikai,3,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Yuan Shikai],[Sino-Tibetan relations during the Ming dynast...,[{'content': 'Answer the question based on the...,Yuan Shikai,1,1.0
2,Tualatin Mountains,3,[{'content': 'Answer the question based on the...,"[{'id': 5, 'is_supporting': True, 'text': '# P...",[Tualatin Mountains],"[Portland, Oregon, Coles Creek (Pennsylvania),...",[{'content': 'Answer the question based on the...,Tualatin Mountains,1,1.0
3,Casa Loma,3,[{'content': 'Answer the question based on the...,"[{'id': 1, 'is_supporting': False, 'text': '# ...",[Casa Loma],"[Arna Selznick, Casa Loma, Natalie Turner]",[{'content': 'Answer the question based on the...,Marksburg Castle,0,0.0
4,1941,3,[{'content': 'Answer the question based on the...,"[{'id': 2, 'is_supporting': False, 'text': '# ...",[1941],"[Pacific War, Bang Bon District, Ercole Manfredi]",[{'content': 'Answer the question based on the...,1941,1,1.0


In [21]:
df[['exact_match', 'f1']].mean()

exact_match    0.095000
f1             0.148418
dtype: float64